# Task 1A: Financial Data Pipeline
Dynamic two-year daily NVDA OHLCV, pandas/numpy indicators, real recent news and deterministic momentum. No LLM is used.

Run all code cells in order. Supply your actual public repository URL below in Colab if no checkout exists. Outputs have deliberately not been prepopulated. Yahoo failures and headline shortfalls are reported explicitly.

In [ ]:
import subprocess
import sys
from pathlib import Path

REPOSITORY_URL = "https://github.com/Randimal/CDAZZDEV-MLE-Poshitha-Malagala.git"
cwd = Path.cwd()
repo_root = next(
    (
        p
        for p in (cwd, *cwd.parents)
        if (p / "task1_financial" / "data_pipeline.py").exists()
    ),
    None,
)
if repo_root is None:
    if not REPOSITORY_URL.startswith("https://github.com/"):
        raise ValueError(
            "Set REPOSITORY_URL to your public GitHub repository URL and rerun."
        )
    repo_root = cwd / "CDAZZDEV-MLE-Poshitha-Malagala"
    if not repo_root.exists():
        subprocess.run(
            ["git", "clone", "--", REPOSITORY_URL, str(repo_root)], check=True
        )
    if not (repo_root / "task1_financial" / "data_pipeline.py").exists():
        raise ValueError("Checkout does not contain the Phase 1 implementation.")
subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "-r",
        str(repo_root / "requirements.txt"),
    ],
    check=True,
)
sys.path.insert(0, str(repo_root))

## Fetch daily adjusted prices
Metadata/news failures degrade gracefully. An essential price failure produces a domain error; retry the fetch cell later if Yahoo is unavailable.

In [ ]:
from dataclasses import asdict

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

from shared.config import PipelineConfig
from shared.logging_utils import configure_logging
from task1_financial.data_pipeline import run_pipeline
from task1_financial.models import PipelineError

configure_logging()
config = PipelineConfig(ticker="NVDA", period="2y", news_count=10)
result = None
try:
    result = run_pipeline(config)
except PipelineError as exc:
    print(f"Pipeline unavailable: {exc}. Retry when data/network is available.")

In [ ]:
if result is not None:
    print("Recent daily OHLCV")
    display(result.data[["Open", "High", "Low", "Close", "Volume"]].tail())
    indicators = [
        "SMA_50",
        "SMA_200",
        "RSI",
        "MACD",
        "MACD_signal",
        "MACD_histogram",
        "BB_middle",
        "BB_upper",
        "BB_lower",
    ]
    print("Latest technical indicators")
    display(result.data[indicators].tail(1).T)
    print("Summary dictionary")
    display(result.summary)
    print("Momentum evidence")
    display(asdict(result.momentum))

In [ ]:
if result is not None:
    print(f"{len(result.news)} usable recent headlines (requested {config.news_count})")
    display(
        pd.DataFrame(
            [asdict(item) for item in result.news],
            columns=["title", "publisher", "published_at", "url"],
        )
    )
    if len(result.news) < config.news_count:
        print("Yahoo supplied fewer usable headlines; no headlines were fabricated.")

In [ ]:
if result is not None:
    data = result.data
    bands = data.dropna(subset=["BB_upper", "BB_middle", "BB_lower"])
    checks = {
        "at_least_200_observations": len(data) >= 200,
        "latest_SMA200_defined": pd.notna(data["SMA_200"].iloc[-1]),
        "RSI_in_range": bool(data["RSI"].dropna().between(0, 100).all()),
        "bands_ordered": bool(
            (
                (bands.BB_upper >= bands.BB_middle)
                & (bands.BB_middle >= bands.BB_lower)
            ).all()
        ),
        "valid_momentum": result.momentum.signal in {"BULLISH", "BEARISH", "NEUTRAL"},
    }
    display(checks)
    assert all(checks.values()), "Sanity check failed"
    print(
        f"Requested {config.period}; received {data.index.min()} through {data.index.max()}"
    )

In [ ]:
if result is not None:
    fig, ax = plt.subplots(figsize=(12, 5))
    result.data[["Close", "SMA_50", "SMA_200"]].plot(ax=ax)
    ax.set(
        title=f"{result.ticker}: adjusted daily Close and moving averages",
        xlabel="Date",
        ylabel="Adjusted price (quote currency)",
    )
    ax.grid(alpha=0.25)
    fig.tight_layout()
    plt.show()

## Interpretation and limitations
Current price means the latest adjusted daily close. YTD uses the last close before January 1; missing baseline yields null. Bands use population standard deviation (ddof=0). RSI uses an arithmetic Wilder seed followed by alpha=1/14 smoothing. Momentum is a documented five-vote heuristic. See the Task 1 README for formulas, warm-up conventions and error policy.

## Task 1B: Groq sentiment and recommendation
These cells make real Groq calls: one per headline and up to three for the recommendation. Use a Groq model supporting JSON object mode. API keys are read from environment variables, Colab Secrets (userdata), or a hidden input prompt. Set GROQ_MODEL in the environment or Colab Secrets, or enter the model identifier when prompted. Never paste keys into notebook source or print environment variables.

A failed headline is counted rather than replaced; no valid sentiments means aggregate unavailable. Failed recommendation retries do not imply HOLD.

In [ ]:
import getpass
import os

from task1_financial.llm import GroqClient, LLMConfigurationError
from task1_financial.recommendation import get_recommendation
from task1_financial.sentiment import analyze_headlines


def load_groq_settings():
    for name in ("GROQ_API_KEY", "GROQ_MODEL"):
        if not os.environ.get(name, "").strip():
            try:
                from google.colab import userdata

                value = userdata.get(name)
                if value:
                    os.environ[name] = value.strip()
            except Exception:
                pass  # Colab absent, secret absent, or permission not granted.
    if not os.environ.get("GROQ_API_KEY", "").strip():
        os.environ["GROQ_API_KEY"] = getpass.getpass("Groq API key (hidden): ").strip()
    if not os.environ.get("GROQ_MODEL", "").strip():
        os.environ["GROQ_MODEL"] = input("Groq JSON-capable model identifier: ").strip()


load_groq_settings()
groq_client = None
try:
    groq_client = GroqClient()
    print("Groq client configured. Credentials are not displayed.")
except LLMConfigurationError:
    print("Set GROQ_API_KEY and GROQ_MODEL, then rerun this cell.")
except Exception:
    print(
        "Groq client initialization failed; check configuration without sharing secrets."
    )

In [ ]:
sentiment_batch = None
if result is not None and groq_client is not None:
    sentiment_batch = analyze_headlines(result.ticker, result.news, groq_client)
    display(
        pd.DataFrame(
            [item.model_dump() for item in sentiment_batch.results],
            columns=["headline", "sentiment", "confidence", "brief_reason"],
        )
    )
    print("Aggregate sentiment (successful responses only)")
    display(sentiment_batch.aggregate.model_dump())
else:
    print("Fetch valid Task 1A data and configure Groq before sentiment analysis.")

### Validated technical recommendation
The payload includes all nine latest indicators, adjusted current price, deterministic momentum and aggregate sentiment with success/failure counts. Missing numeric values become JSON null. The prompt asks for interactions/conflicts rather than numerical restatement and forbids invented crossovers. Reasoning must contain 3–5 complete sentences. Three bounded attempts include API and validation failures.

In [ ]:
recommendation = None
if result is not None and groq_client is not None:
    aggregate = sentiment_batch.aggregate if sentiment_batch is not None else None
    recommendation = get_recommendation(result, groq_client, aggregate)
    if recommendation is not None:
        display(recommendation.model_dump())
    else:
        print(
            "No valid LLM recommendation after bounded retries; no signal substituted."
        )
else:
    print("Valid Task 1A data and Groq configuration are required.")

### Bonus: equity research brief
Create Markdown, print-styled HTML and a Close/SMA50/SMA200 PNG chart under task1_financial/outputs/. The HTML is designed as a compact A4 brief; browser print scaling and fonts can affect pagination. It uses real pipeline data and marks unavailable analysis explicitly. Reports and notebook output should be reviewed before submission; no example artifacts are fabricated.

In [ ]:
from pathlib import Path

from IPython.display import FileLink, Image

from task1_financial.report import generate_report

report_artifacts = None
if result is not None:
    aggregate = sentiment_batch.aggregate if sentiment_batch is not None else None
    report_artifacts = generate_report(
        result,
        aggregate,
        recommendation,
        Path(repo_root) / "task1_financial" / "outputs",
    )
    display(Image(filename=str(report_artifacts.chart)))
    for artifact in (
        report_artifacts.markdown,
        report_artifacts.html,
        report_artifacts.chart,
    ):
        print(artifact.relative_to(repo_root))
        display(FileLink(str(artifact)))
else:
    print("No valid price data: no research report generated.")

Download the HTML together with its PNG chart to preserve the relative image reference. No report files are committed automatically. The brief is educational research and the LLM output is not a calibrated trading strategy. Task 2 remains pending; Task 3 is demonstrated in its own notebook.